# nb05_robustness
**Chủ:** C · tiêu chí 7 (đánh giá). Đề này không có mô hình phân lớp nên **không có train/test, accuracy hay ROC**. Đánh giá = baseline + khoảng tin cậy + kết luận giữ/đổi + phân tích lỗi. Đơn vị luôn là APP.

> **Ngưỡng "đổi" ở ô thứ hai phải được ghi vào `docs/decision_log.md` TRƯỚC khi chạy notebook này.** Đổi ngưỡng sau khi thấy kết quả là chọn kết quả theo ý mình.

Bảy phép thử: (1) khử trùng A0/A1/A2, (2) `min_store_reviews` = 0/50/500, (3) trung vị thay trung bình, (4) bootstrap 95% theo app, (5) baseline toàn cục + hoán vị nhãn danh mục (1 000 lần), (6) độ ổn định xếp hạng (Spearman, trùng top-5/bottom-5), (7) tỉ lệ hiếm (Wilson, ô nhỏ).

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm
from sqlalchemy import create_engine, text
from config import *
from dmutils import save_result

pd.set_option("display.width", 170, "display.max_columns", 40)
ENGINE = create_engine(DB_URL)
FIG_DIR = OUT_DIR / "figures"; FIG_DIR.mkdir(parents=True, exist_ok=True)

def q(sql):
    """Chạy một câu SQL, trả DataFrame."""
    return pd.read_sql(text(sql), ENGINE)

def rec(df):
    """DataFrame -> list dict an toàn cho JSON (NaN -> None, categorical -> str)."""
    d = df.copy()
    for c in d.columns:
        if str(d[c].dtype) == "category": d[c] = d[c].astype(str)
    d = d.astype(object).where(d.notna(), None)
    return d.to_dict("records")

def wilson(k, n, alpha=0.05):
    """Khoảng Wilson cho tỉ lệ k/n -> (rate, lo, hi). Cùng công thức với mart.wilson_lo/hi."""
    if n == 0: return (np.nan, np.nan, np.nan)
    z = norm.ppf(1 - alpha / 2); p = k / n; den = 1 + z**2 / n
    ctr = (p + z**2 / (2 * n)) / den
    half = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / den
    return (p, max(0.0, ctr - half), min(1.0, ctr + half))

def low_table(df, by, order=None):
    """Bảng tỉ lệ app điểm thấp theo nhóm: n, k, rate, Wilson lo/hi, enough_n (n >= MIN_CELL_N)."""
    by = [by] if isinstance(by, str) else list(by)
    keys = by + ([order] if order else [])
    g = df.groupby(keys, observed=True)["is_low"].agg(n="size", k="sum").reset_index()
    w = pd.DataFrame([wilson(int(k), int(n)) for k, n in zip(g["k"], g["n"])], columns=["rate", "lo", "hi"])
    g = pd.concat([g, w], axis=1)
    g["enough_n"] = g["n"] >= MIN_CELL_N
    return g.sort_values(order if order else "rate", ascending=bool(order)).reset_index(drop=True)

from scipy.stats import spearmanr

# ---- Ngưỡng chốt trước (ghi decision_log trước khi chạy) ----
RHO_MIN = 0.90          # Spearman xếp hạng danh mục so với cấu hình tham chiếu < 0.90 => thứ hạng "đổi"
TOP5_MAX_CHANGE = 1     # số danh mục đổi trong top-5/bottom-5 >= 2 => "đổi"
MEDIAN_GAP = 0.10       # |trung bình - trung vị| > 0.10 điểm => đáng chú ý
EFFECT_TOL_PP = 2.0     # hiệu ứng lệch tham chiếu > 2 điểm % (hoặc đổi dấu) => "đổi"
NEGLIGIBLE_PP = 1.0     # |hiệu ứng| < 1 điểm % ở MỌI cấu hình => coi là "không có hiệu ứng đáng kể" (không xét đổi dấu)
N_BOOT, N_PERM = 2000, 1000
rng = np.random.default_rng(RANDOM_STATE)

## 0. Ba phiên bản dữ liệu dựng từ `drop_reason` trong `stg.app_all_rows`
- **A0** = bỏ dòng lệch cột (`misaligned`)
- **A1** = A0 + bỏ dòng trùng hoàn toàn (`exact_dup`)
- **A2** = A1 + một dòng mỗi tên (`kept = true`, bản chính)

Lưu ý: ở A0/A1 một app có thể xuất hiện nhiều lần (cùng tên khác thuộc tính); đó chính là điều phép thử 1 đo.

In [ ]:
stg = q("""
SELECT src_row_id, app_name_norm, category, rating, store_reviews, price, last_updated, kept,
       COALESCE(drop_reason, '') AS drop_reason
FROM stg.app_all_rows
""")
stg["kept"] = stg["kept"].astype(str).str.lower().isin(["true", "t", "1"])     # an toàn với mọi driver
stg["last_updated"] = pd.to_datetime(stg["last_updated"])
REF_DATE = stg.loc[stg["kept"], "last_updated"].max()        # ngày lớn nhất trong dữ liệu, giống dw.dim_date
stg["age_days"] = (REF_DATE - stg["last_updated"]).dt.days
A0 = stg[stg["drop_reason"] != "misaligned"]
A1 = A0[A0["drop_reason"] != "exact_dup"]
A2 = stg[stg["kept"]]
VERS = {"A0": A0, "A1": A1, "A2": A2}
for k, d in VERS.items(): print(k, "dòng:", len(d), "| tên duy nhất:", d["app_name_norm"].nunique())
print("ngày tham chiếu độ cũ:", REF_DATE.date())

def prep(d, min_rev):
    x = d[d["rating"].notna() & (d["store_reviews"] >= min_rev)].copy()
    x["is_low"] = x["rating"] <= LOW_RATING_THRESHOLD
    x["price_type"] = np.where(x["price"] > 0, "Paid", "Free")
    return x

def cat_rates(d, min_rev):
    return low_table(prep(d, min_rev), "category").set_index("category")

def effect(d, min_rev):
    """3 kết luận của nb03 dưới dạng số: (a) danh mục đứng đầu, (b) Paid - Free, (c) app cũ (>2y) - app mới (<=3m), đơn vị điểm %."""
    x = prep(d, min_rev)
    t = low_table(x, "category"); t = t[t["enough_n"]]
    paid = x.loc[x.price_type == "Paid", "is_low"].mean() - x.loc[x.price_type == "Free", "is_low"].mean()
    old = x.loc[x.age_days > 730, "is_low"].mean() - x.loc[x.age_days <= 90, "is_low"].mean()
    return {"top_category": t.sort_values("rate").iloc[-1]["category"], "paid_minus_free_pp": 100 * paid, "old_minus_new_pp": 100 * old}

# tính nhất quán với dw: A2 + ngưỡng chính phải ra đúng n_analysis của nb03
n_check = len(prep(A2, MIN_STORE_REVIEWS))
n_dw = int(q("SELECT COUNT(*) AS n FROM dw.fact_app WHERE rating IS NOT NULL AND store_reviews >= %d" % MIN_STORE_REVIEWS).n[0])
assert n_check == n_dw, (n_check, n_dw)
print("A2 khớp dw.fact_app:", n_check)

## Phép 1 + 2 + 6. Khử trùng × ngưỡng (bảng 3×3): xếp hạng danh mục có đổi không?

In [ ]:
grid = {(v, m): cat_rates(d, m) for v, d in VERS.items() for m in (0, MIN_STORE_REVIEWS, 500)}
refr = grid[("A2", MIN_STORE_REVIEWS)]; ref_ok = refr[refr["enough_n"]]
rows = []
for (v, m), t in grid.items():
    common = ref_ok.index.intersection(t[t["enough_n"]].index)
    rho = spearmanr(ref_ok.loc[common, "rate"], t.loc[common, "rate"]).statistic
    top5 = set(ref_ok["rate"].nlargest(5).index) & set(t.loc[common, "rate"].nlargest(5).index)
    bot5 = set(ref_ok["rate"].nsmallest(5).index) & set(t.loc[common, "rate"].nsmallest(5).index)
    rows.append({"version": v, "min_reviews": m, "n_rows": int(t["n"].sum()), "n_categories_common": len(common),
                 "spearman_vs_ref": rho, "top5_kept": len(top5), "bottom5_kept": len(bot5), **effect(VERS[v], m)})
robust = pd.DataFrame(rows)
robust["rank_changed"] = ((robust["spearman_vs_ref"] < RHO_MIN) | ((5 - robust["top5_kept"]) > TOP5_MAX_CHANGE)
                          | ((5 - robust["bottom5_kept"]) > TOP5_MAX_CHANGE))
print(robust.round(3).to_string(index=False))
save_result("nb05_robustness_grid", rec(robust))

## Phép 3. Trung vị thay trung bình

In [ ]:
x = prep(A2, MIN_STORE_REVIEWS)
mm = x.groupby("category")["rating"].agg(n="size", mean="mean", median="median")
mm = mm[mm["n"] >= MIN_CELL_N]; mm["gap"] = (mm["mean"] - mm["median"]).abs()
median_report = {"spearman_mean_vs_median": float(spearmanr(mm["mean"].rank(), mm["median"].rank()).statistic),
                 "n_categories": len(mm), "n_gap_above_threshold": int((mm["gap"] > MEDIAN_GAP).sum()),
                 "share_exact_5": float((x["rating"] == 5.0).mean()), "n": len(x)}
print(median_report); save_result("nb05_median_vs_mean", median_report)

## Phép 4. Bootstrap 95%, resample theo APP (mỗi dòng của A2 đã là một app)

In [ ]:
low, cats = x["is_low"].to_numpy(), x["category"].to_numpy()
ci_rows = []
for c in sorted(set(cats)):
    ix = np.where(cats == c)[0]
    if len(ix) < MIN_CELL_N: continue
    draws = low[ix][rng.integers(0, len(ix), size=(N_BOOT, len(ix)))].mean(axis=1)
    lo, hi = np.percentile(draws, [2.5, 97.5]); p, wlo, whi = wilson(int(low[ix].sum()), len(ix))
    ci_rows.append({"category": c, "n": len(ix), "rate": p, "boot_lo": lo, "boot_hi": hi, "wilson_lo": wlo, "wilson_hi": whi})
boot = pd.DataFrame(ci_rows)
glob = float(low.mean())
boot["vs_global"] = np.where(boot["wilson_lo"] > glob, "cao hơn", np.where(boot["wilson_hi"] < glob, "thấp hơn", "không khác"))
print("baseline 1: tỉ lệ toàn cục =", round(glob, 4), "| n =", len(x)); print(boot["vs_global"].value_counts().to_string())
save_result("nb05_bootstrap_by_category", rec(boot)); save_result("nb05_global_rate", {"low_rate_global": glob, "n": len(x)})

## Phép 5. Baseline hoán vị nhãn danh mục
Nếu nhãn danh mục không mang thông tin, chênh lệch cao–thấp giữa các danh mục vẫn xuất hiện do nhiễu. So chênh lệch quan sát với phân phối khi xáo nhãn 1 000 lần.
Chỉ dùng danh mục đủ n (tránh ô nhỏ tự tạo dải rộng). Đây là phân tích **khám phá**: p-value không dùng để "khẳng định", báo kèm kích thước chênh lệch.

In [ ]:
keep = np.isin(cats, boot["category"].to_numpy()); lw, cs = low[keep], cats[keep]
obs = pd.Series(lw).groupby(cs).mean(); obs_range = float(obs.max() - obs.min())
null = np.empty(N_PERM)
for i in range(N_PERM):
    r = pd.Series(lw).groupby(rng.permutation(cs)).mean(); null[i] = r.max() - r.min()
perm = {"obs_range_pp": 100 * obs_range, "null_median_pp": 100 * float(np.median(null)),
        "null_p95_pp": 100 * float(np.percentile(null, 95)), "p_value_exploratory": float((np.sum(null >= obs_range) + 1) / (N_PERM + 1)),
        "n": int(keep.sum()), "n_categories": int(len(obs)), "n_perm": N_PERM}
print(perm); save_result("nb05_permutation", perm)

## Phép 7. Tỉ lệ hiếm và ô nhỏ

In [ ]:
allc = cat_rates(A2, MIN_STORE_REVIEWS); small = allc[~allc["enough_n"]]
rare = {"n_categories_total": len(allc), "n_categories_below_min_cell": len(small),
        "n_apps_in_small_categories": int(small["n"].sum()), "small_categories": list(small.index)}
print(rare); save_result("nb05_rare_cells", rare)

## Bảng "kết luận nào giữ, nào đổi, đổi bao nhiêu"

In [ ]:
refe = robust[(robust.version == "A2") & (robust.min_reviews == MIN_STORE_REVIEWS)].iloc[0]
def verdict(col, kind):
    vals = robust[col]
    if kind == "cat":
        n_diff = int((vals != refe[col]).sum())
        return ("Giữ" if n_diff == 0 else "Đổi"), f"{n_diff}/9 cấu hình đổi danh mục đứng đầu"
    if bool((vals.abs() < NEGLIGIBLE_PP).all()):
        return "Giữ", f"hiệu ứng ≈ 0 ở cả 9 cấu hình (|hiệu ứng| < {NEGLIGIBLE_PP} điểm %): kết luận là 'không khác biệt đáng kể'"
    same_sign = bool((np.sign(vals) == np.sign(refe[col])).all())
    spread = float((vals - refe[col]).abs().max())
    return ("Giữ" if same_sign and spread <= EFFECT_TOL_PP else "Đổi"), f"dấu {'ổn định' if same_sign else 'ĐỔI'}, lệch tối đa {spread:.1f} điểm %"
conc = []
for name, col, kind in [("Danh mục có tỉ lệ app điểm thấp cao nhất", "top_category", "cat"),
                        ("App trả phí khác app miễn phí", "paid_minus_free_pp", "num"),
                        ("App cũ (>2 năm) khác app mới (<=3 tháng)", "old_minus_new_pp", "num")]:
    vd, why = verdict(col, kind); conc.append({"ket_luan": name, "tham_chieu": refe[col], "giu_hay_doi": vd, "chi_tiet": why})
conclusions = pd.DataFrame(conc); print(conclusions.to_string(index=False)); save_result("nb05_conclusions", rec(conclusions))

## Phân tích lỗi (C viết bằng lời SAU khi xem kết quả, 5–8 dòng)
Trả lời: (1) danh mục nào nhạy nhất với `min_store_reviews` và vì sao (thường vì danh mục nhỏ, hoặc nhiều app 5.0 ít đánh giá); (2) khử trùng làm đổi kết luận ở đâu; (3) những ô nào bị bỏ vì `n < MIN_CELL_N` và chiếm bao nhiêu app; (4) hoán vị cho thấy chênh lệch giữa danh mục có vượt nhiễu không, và vì sao p-value chỉ mang tính khám phá. Đây là "error analysis" của đề này.

## C6. Số liệu đạo đức: xếp hạng thô so với xếp hạng có ngưỡng (C cung cấp số, D diễn giải trong `docs/ethics.md`)

In [ ]:
r = q("SELECT c.category, f.app_name, f.rating, f.store_reviews FROM dw.fact_app f JOIN dw.dim_category c USING (category_key) WHERE f.rating IS NOT NULL")
r["raw_rank"] = r.groupby("category")["rating"].rank(method="min", ascending=False)       # thô: chỉ theo điểm, hoà cùng hạng
top_raw = r[r["raw_rank"] <= 10]
five = r[r["rating"] == 5.0]
filt = r[r["store_reviews"] >= MIN_STORE_REVIEWS].copy()
filt["thr_rank"] = filt.groupby("category")["rating"].rank(method="min", ascending=False)
chg = []
for c, g in r.groupby("category"):
    r10 = set(g.loc[g["raw_rank"] <= 10, "app_name"]); f = filt[filt["category"] == c]
    t10 = set(f.loc[f["thr_rank"] <= 10, "app_name"])
    chg.append({"category": c, "n_rated": len(g), "n_after_threshold": len(f), "n_top10_raw": len(r10),
                "n_top10_raw_thin": int((g.loc[g["raw_rank"] <= 10, "store_reviews"] < MIN_STORE_REVIEWS).sum()),
                "n_in_both_top10": len(r10 & t10)})
rank_by_cat = pd.DataFrame(chg)
ethics = {"n_rated": len(r), "n_top_raw": len(top_raw), "n_top_raw_thin": int((top_raw["store_reviews"] < MIN_STORE_REVIEWS).sum()),
          "share_thin_in_raw_top10": float((top_raw["store_reviews"] < MIN_STORE_REVIEWS).mean()),
          "n_exact_5": len(five), "n_exact_5_thin": int((five["store_reviews"] < MIN_STORE_REVIEWS).sum()),
          "share_exact_5_thin": float((five["store_reviews"] < MIN_STORE_REVIEWS).mean()),
          "min_store_reviews": MIN_STORE_REVIEWS}
print(ethics); print(rank_by_cat.sort_values("n_top10_raw_thin", ascending=False).head(8).to_string(index=False))
save_result("nb05_ethics_rank_bias", ethics); save_result("nb05_ethics_rank_by_category", rec(rank_by_cat))

## Việc cuối cùng của C trước khi nộp
- Chạy lại cả notebook hai lần, so `outputs/results_parts/nb05_*.json` phải giống hệt (mọi chỗ ngẫu nhiên đã dùng `RANDOM_STATE`).
- Mỗi kết luận trong báo cáo có một dòng trong bảng "giữ/đổi" ở trên.